<a href="https://colab.research.google.com/github/alexcompose/qwen_2agent_chain/blob/main/qwen_2agent_chain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain langchain-core langchain-huggingface transformers torch accelerate langgraph


In [ ]:
from langchain_huggingface import HuggingFacePipeline
from langchain_core.messages import HumanMessage, SystemMessage
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langgraph.graph import StateGraph, END
from typing import TypedDict, List


In [ ]:
# 1. Load local Qwen model into pipeline
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto")

hf_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,
    temperature=0.2
)


In [ ]:
# 2. Define state structure for the agents
class AgentState(TypedDict):
    input: str
    draft: str
    critique: str
    final_output: str

# 3. Define Agent Nodes
def writer_agent(state: AgentState):
    messages = [
        {"role": "system", "content": "You are a precise technical writer. Write a clear, concise response to the user's prompt."},
        {"role": "user", "content": state["input"]}
    ]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    response = hf_pipeline(prompt_text, max_new_tokens=128, do_sample=True, temperature=0.2)

    # Strip the prompt from the generated output
    generated = response[0]["generated_text"]
    draft_text = generated[len(prompt_text):].strip() if generated.startswith(prompt_text) else generated
    return {"draft": draft_text}

def critic_agent(state: AgentState):
    messages = [
        {"role": "system", "content": "You are a strict technical editor. Your job is to aggressively clean up the draft, fix structural awkwardness, and output a polished final version without any meta-talk."},
        {"role": "user", "content": f"Original Request: {state['input']}\n\nDraft to fix:\n{state['draft']}"}
    ]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    response = hf_pipeline(prompt_text, max_new_tokens=128, do_sample=True, temperature=0.2)

    generated = response[0]["generated_text"]
    critique_text = generated[len(prompt_text):].strip() if generated.startswith(prompt_text) else generated
    return {"final_output": critique_text}

In [ ]:
# 4. Compile the 2-agent graph
workflow = StateGraph(AgentState)
workflow.add_node("writer", writer_agent)
workflow.add_node("critic", critic_agent)

workflow.set_entry_point("writer")
workflow.add_edge("writer", "critic")
workflow.add_edge("critic", END)

app = workflow.compile()


In [ ]:

# 5. Run the multi-agent pipeline
initial_state = {"input": "Explain why would python be preferred in competitive programming."}
result = app.invoke(initial_state)

print("--- FINAL AGENT OUTPUT ---")
print(result["final_output"])